In [6]:
import numpy as np

def apply_operators(det_idx, orb_idxs, types):
    # Convert determinant index to binary representation
    det = det_idx
    phase = 1
    
    # Apply operators in sequence
    for orb_idx, op_type in zip(orb_idxs, types):
        if op_type == 1:  # Creation operator
            # Check if orbital is already occupied
            if det & (1 << orb_idx):
                return None  # Pauli exclusion principle
            # Count occupied orbitals before this one for phase
            phase *= -1 if bin(det & ((1 << orb_idx) - 1)).count('1') % 2 == 1 else 1
            det |= (1 << orb_idx)
        else:  # Annihilation operator (op_type == 0)
            # Check if orbital is occupied
            if not (det & (1 << orb_idx)):
                return None  # Can't annihilate empty orbital
            # Count occupied orbitals before this one for phase
            phase *= -1 if bin(det & ((1 << orb_idx) - 1)).count('1') % 2 == 1 else 1
            det &= ~(1 << orb_idx)
    
    return det, phase


def op(orb_idxs, types, ci):
    # apply a sequence of creation and annihilation operators defined by orb_idxs and types to the ci vector
    # types: 1 for creation, 0 for annihilation
    # ci: 2^n vector for n spin orbitals, the back is alpha front is beta
    # for example the 0111 corresponds to alpha 11, beta 01
    ci_new = np.zeros_like(ci)
    for det_idx in range(len(ci)):
        if ci[det_idx] != 0:
            det_result = apply_operators(det_idx, orb_idxs, types)
            if det_result is not None:
                det_new, phase = det_result
                ci_new[det_new] += phase * ci[det_idx]
    return ci_new

ci = np.zeros(16)
ci[3] = 1  # corresponds to alpha 11, beta 00
orb_idxs = [0, 2]
types = [0, 1]  
ci_new = op(orb_idxs, types, ci)
print(ci_new)

[ 0.  0.  0.  0.  0.  0. -1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
